In [0]:
%sql
/* check for distinct results in the silver layer */
select distinct Result from rugby_data_dev.rugby_silver.match_results_notebook

In [0]:
%sql
/* check for distinct seasons in the silver layer */
select distinct Season from rugby_data_dev.rugby_silver.match_results_notebook

In [0]:
%sql
/* check for distinct team names in the silver layer */
select HomeTeam as Team
from rugby_data_dev.rugby_silver.match_results_notebook
union
select AwayTeam as Team
from rugby_data_dev.rugby_silver.match_results_notebook
order by Team;
    


In [0]:
%sql 
/* check for distinct rounds */
select distinct Round from rugby_data_dev.rugby_silver.match_results_notebook
order by Round asc;

In [0]:
%sql
/* create the dimensions and fact tables */

-- Dimension tables
create or replace table rugby_data_dev.rugby_gold.Result_Dim as 
select row_number() over (order by Result) as Result_Id, Result
from (
  select distinct Result 
  from rugby_data_dev.rugby_silver.match_results_notebook
);

create or replace table rugby_data_dev.rugby_gold.Season_Dim as 
select row_number() over (order by Season) as Season_Id, Season
from (
  select distinct Season 
  from rugby_data_dev.rugby_silver.match_results_notebook
);

create or replace table rugby_data_dev.rugby_gold.Teams_Dim (
  select row_number() over (order by HomeTeam) as Team_Id, HomeTeam as Team_Name
  from (
    select distinct HomeTeam 
    from rugby_data_dev.rugby_silver.match_results_notebook
  )
);

create or replace table rugby_data_dev.rugby_gold.Round_Dim (
  select row_number() over (order by Round) as Round_Id, Round
  from (
    select distinct Round 
    from rugby_data_dev.rugby_silver.match_results_notebook
  )
);

-- Fact table
create or replace table rugby_data_dev.rugby_gold.Match_Fact as 
select 
  mr.MatchId,
  htd.Team_Id as Home_Team_Id,
  atd.Team_Id as Away_Team_Id,
  sd.Season_Id as Season_Id,
  rod.Round_Id as Round_Id,
  mr.HomeScore as Home_Score,
  mr.AwayScore as Away_Score,
  red.Result_Id as Result_Id,
  er.HomeEloBefore as Home_Elo_Before,
  er.HomeEloAfter as Home_Elo_After,
  er.HomeEloChange as Home_Elo_Change,
  er.AwayEloBefore as Away_Elo_Before,
  er.AwayEloAfter as Away_Elo_After,
  er.AwayEloChange as Away_Elo_Change,
  mr.MatchPointsDifference as Match_Points_Difference,
  mr.HomePointsDifference as Home_Points_Difference,
  mr.AwayPointsDifference as Away_Points_Difference,
--add meta data columns
  current_timestamp() as last_upload,
  'gold_layer' as pipelineStage

from rugby_data_dev.rugby_silver.match_results_notebook as mr 
-- joins the elo ratings table and all dimension tables
left join rugby_data_dev.rugby_gold.elo_ratings as er on mr.MatchId = er.MatchId
left join rugby_data_dev.rugby_gold.Result_Dim as red on mr.Result = red.Result
left join rugby_data_dev.rugby_gold.Season_Dim as sd on mr.Season = sd.Season
left join rugby_data_dev.rugby_gold.Teams_Dim as htd on mr.HomeTeam = htd.Team_Name
left join rugby_data_dev.rugby_gold.Teams_Dim as atd on mr.AwayTeam = atd.Team_Name
left join rugby_data_dev.rugby_gold.Round_Dim as rod on mr.Round = rod.Round


/* Old Schema */

-- Create table Result_Dim (
--   Result_Id Int primary key,
--   Result varchar(8)
-- );

-- create table Season_Dim (
--   Season_Id Int primary key,
--   Season varchar(5)
-- );

-- create table Teams_Dim (
--   Team_Id Int primary key,
--   Team_Name varchar(25)
-- );

-- create table Round_Dim (
--   Round_Id Int primary key,
--   Round varchar(2)
-- );

-- create table Match_Fact (
--   Match_Id Int primary key,
--   Home_Team_Id Int,
--   Away_Team_Id Int,
--   Season_Id Int,
--   Round_Id Int,
--   Home_Score Int,
--   Away_Score Int,
--   Result_Id Int,
--   Home_Elo_Before Int,
--   Home_Elo_After Int,
--   Home_Elo_Change Int,
--   Away_Elo_Before Int,
--   Away_Elo_After Int,
--   Away_Elo_Change Int,
--   Match_Points_Difference Int,
--   Home_Points_Difference Int,
--   Away_Points_Difference Int,

--   foreign key (Home_Team_Id) references Teams_Dim(Team_Id),
--   foreign key (Away_Team_Id) references Teams_Dim(Team_Id),
--   foreign key (Season_Id) references Season_Dim(Season_Id),
--   foreign key (Round_Id) references Round_Dim(Round_Id),
--   foreign key (Result_Id) references Result_Dim(Result_Id)

-- );